# Week 7 – Roll A & B (CSV-põhine)

Andmed loetakse ainult lokaalsetest CSV failidest (`sales.csv`, `customers.csv`).
Lõpptulemus on puhastatud `df`

## Roll A – Andmete laadimine

Laadin `sales` ja `customers` CSV failidest pandas DataFrame'i, kontrolli struktuuri ja liida tabelid `customer_id` põhjal.


In [8]:
import pandas as pd

# Loeme andmed lokaalsetest CSV failidest (encoding='utf-8-sig' eemaldab BOM-i).
df_sales = pd.read_csv("sales.csv", encoding="utf-8-sig")
df_customers = pd.read_csv("customers.csv", encoding="utf-8-sig")

print("Sales shape:", df_sales.shape)
display(df_sales.head())

print("Customers shape:", df_customers.shape)
display(df_customers.head())

print("\nSales dtypes:")
print(df_sales.dtypes)

Sales shape: (15234, 11)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


Customers shape: (3150, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2001,Eha,Aas,eha.aas@telia.ee,+372 8713 1455,Tallinn,2024-02-27,NaN,1973
1,2002,Aivar,Kõiv,aivar.koiv@outlook.com,+372 8943 8684,Haapsalu,2025-01-09,bronze,1988
2,2003,Maris,Rebane,maris.rebane@telia.ee,+372 5918 5726,Tartu,2021-02-03,NaN,1999
3,2004,Jaak,Talvik,jaak.talvik@mail.ee,+372 8554 4232,Tallinn,2023-11-12,silver,1974
4,2005,Raivo,Koppel,raivo.koppel@yahoo.com,+372 5298 4365,Tallinn,2023-05-22,bronze,2004



Sales dtypes:
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object


In [9]:
# Liidan tabelid customer_id põhjal (left join säilitab kõik müügiread).
df = pd.merge(df_sales, df_customers, on="customer_id", how="left")

print("Liidetud (merged) shape:", df.shape)
print("\nLiidetud dtypes:")
print(df.dtypes)
display(df.head())

# Kontroll: kas võtmeveerud on olemas?
vajalikud = ["customer_id", "sale_date", "total_price", "email"]
print("\nOlemasolevad võtmeveerud:", [c for c in vajalikud if c in df.columns])

Liidetud (merged) shape: (15234, 19)

Liidetud dtypes:
sale_id                int64
invoice_id               str
sale_date                str
customer_id          float64
product_id             int64
quantity               int64
unit_price           float64
total_price          float64
channel                  str
store_location           str
payment_method           str
first_name               str
last_name                str
email                    str
phone                    str
city                     str
registration_date        str
loyalty_tier             str
birth_year           float64
dtype: object


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0



Olemasolevad võtmeveerud: ['customer_id', 'sale_date', 'total_price', 'email']


## Roll B – Andmete puhastamine

**Light clean:** põhipuhastus on duplikaatide eemaldamine primäarvõtme `sale_id` järgi (Supabase'is on `sale_id` primäarvõti) ja kuupäevade parsimine.

NULL `customer_id` ja negatiivsed `total_price` **raporteeritakse, kuid säilitatakse** – NULL customer_id tähendab külalist/sidumata klienti ja negatiivsed on tagastused. Nende eemaldamine muudaks tulemuse Supabase allikast erinevaks.

In [10]:
# 1) Roll A väljund (df). Esialgne shape.
print("Esialgne shape:", df.shape)

# 2) Duplikaadid: eemalda primäarvõtme sale_id järgi 
print("\nTäisrea duplikaadid:", df.duplicated().sum())
print("Korduvad sale_id:", df["sale_id"].duplicated().sum())
df = df.drop_duplicates(subset="sale_id")
print("Shape pärast sale_id dedup:", df.shape)

# 3) NULL väärtused 
print("\nNULL-id:")
print(df.isnull().sum())

# 4) Kuupäevade parsimine (mixed + dayfirst -> nii YYYY-MM-DD kui DD/MM/YYYY).
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce", format="mixed", dayfirst=True)
print("\nsale_date tüüp:", df["sale_date"].dtype, "| NaT väärtusi:", df["sale_date"].isna().sum())

# 5) Outlier'id: negatiivsed total_price (tagastused) – raporteeri, kuid ei eemalda.
print("Negatiivseid total_price:", (df["total_price"] < 0).sum())

# 6) Puhastusraport.
print("\n===== PUHASTUSRAPORT =====")
print("Lõplik shape:", df.shape)
print("Unikaalseid kliente:", df["customer_id"].nunique())
print("Kuupäevavahemik:", df["sale_date"].min().date(), "kuni", df["sale_date"].max().date())

Esialgne shape: (15234, 19)

Täisrea duplikaadid: 4086
Korduvad sale_id: 5116
Shape pärast sale_id dedup: (10118, 19)

NULL-id:
sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64

sale_date tüüp: datetime64[us] | NaT väärtusi: 0
Negatiivseid total_price: 195

===== PUHASTUSRAPORT =====
Lõplik shape: (10118, 19)
Unikaalseid kliente: 2551
Kuupäevavahemik: 2023-01-01 kuni 2026-12-03
